# DocMind — Domain RAG Demo (LangChain docs)

End-to-end demo: ingest → FAISS + cross-encoder → LangGraph agent → Redis cache → Groq generation

- Corpus: 44 LangChain/LangGraph docs → 304 chunks
- Embedding: sentence-transformers/all-MiniLM-L6-v2 (CUDA if available)
- Retrieval: FAISS top-15 → cross-encoder/ms-marco-MiniLM-L-6-v2 rerank top-5
- Agent: LangGraph router (retrieval vs fallback keyword)
- Cache: Redis Cloud
- Generation: Groq openai/gpt-oss-20b (Ollama fallback)
- Tracking: MLflow + W&B


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(f"GPU mem: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# Day 1 — corpus stats
from pathlib import Path
import json
chunks_path = Path("../data/processed/chunks.jsonl") if Path("../data/processed/chunks.jsonl").exists() else Path("data/processed/chunks.jsonl")
if not chunks_path.exists():
    chunks_path = Path("data/processed/chunks.jsonl")
cnt = sum(1 for _ in open(chunks_path, encoding="utf-8"))
print(f"chunks: {cnt}")
print(open(chunks_path, encoding="utf-8").readline()[:200])

In [ ]:
# Day 1 — FAISS + rerank search
from src.embed_index import search
for q in ["What is LangChain?", "How does LangGraph differ from LangChain?", "What is FAISS?", "Explain RAG"]:
    hits = search(q)
    print(f"\nQ: {q}")
    for i,h in enumerate(hits[:3]):
        print(f"  {i+1}. faiss={h['faiss_score']:.3f} rerank={h['rerank_score']:.3f} src={h['source'][:60]}")
        print(f"     {h['text'][:120]}...")

In [ ]:
# Day 2 — retrieve-then-generate (Groq)
from src.generate import rag_answer
for q in ["What is LangChain?", "How to build a chatbot with memory in LangGraph?"]:
    out = rag_answer(q, prefer="groq")
    print(f"\nQ: {q}\nA: {out['answer'][:500]}\nlat: {out['latency_ms']:.0f}ms")

In [ ]:
# Day 3 + 4 — LangGraph agent with Redis cache
from src.agent import answer_question
from src.memory import cache_stats
print("cache keys before:", cache_stats())
for q in ["What is LangChain?", "What is LangChain?", "keyword: vector store"]:
    r = answer_question(q, provider="groq", use_cache=True)
    print(f"\nQ: {q}")
    print(f" route={r['route']} hit={r['cache_hit']} lat={r['latency_ms']:.0f}ms")
    print(f" A: {r['answer'][:400]}...")
print("cache keys after:", cache_stats())

In [ ]:
# Day 5/6 — benchmark summary
import json, pathlib
p = pathlib.Path("data/processed/bench_groq.json")
if not p.exists():
    p = pathlib.Path("../data/processed/bench_groq.json")
if p.exists():
    print(p.read_text())
else:
    print("Run: python -m src.benchmark --provider groq --limit 20")

### Metrics (from bench_groq.json)
- Relevance: 50.0%
- Retrieval precision lift (rerank vs FAISS): -8.5% (corpus-dependent; FAISS alone strong on this small 304-chunk set)
- Latency (Groq openai/gpt-oss-20b): ~1589 ms (incl. retrieval+rerank ~400ms + Groq ~900ms)
- Latency (Ollama): not available locally (no Ollama server) — fallback to Groq
- Cache hit rate: 75% (2nd run, 15/20 cached) — 0% on cold run
- Fallback routing: 0%
- GPU: yes, NVIDIA GeForce GTX 1650 (CUDA 12.4, torch 2.6.0+cu124)

W&B: https://wandb.ai/frozenvictorxi-/domain-rag-agent  
MLflow: `mlflow ui` → experiment `docmind-rag`
